# 19 · What the `cell_registration` schema is, table by table

**Read this one first if you are new to it.** No writes anywhere; every cell is read-only.

### The problem it solves, in one paragraph

A cell you image on Monday and the same cell on Wednesday get **different mask numbers** in
suite2p, because each session is segmented on its own. CellReg looks at the shapes and
positions and decides which mask in session 1 is the same neuron as which mask in session 2.
The answer used to be stored as a single number, `roi_group`, written straight onto
`Segmentation.Mask` — and **overwritten** by whichever CellReg run happened last. Nothing
recorded which run wrote it, from which sessions, with which settings, or how confident it
was. This schema keeps all of that, and never touches `roi_group`.

### The one idea to take away

> A **run** is one execution of CellReg over several sessions.
> A **column** is one of those sessions inside the run.
> A **cell** is one neuron the run claims to have followed across columns.
> A **member** is "this cell appears in this session as this mask".

Everything else is provenance hung off those four words.

In [ ]:
import os
import sys
import pathlib
import datajoint as dj

# ── Which `adamacs` am I importing? ──────────────────────────────────────────
# `adamacs` is installed in development mode in several lab environments pointing at
# ~/adamacs, which does NOT contain the cell_registration code. Importing then fails with
# "cannot import name 'cellreg_plan' from 'adamacs.ingest'" naming that other checkout, and
# the message gives no hint that the wrong repository is on the path. Put THIS repository
# first, then check what actually got imported.
_HERE = pathlib.Path.cwd()
ROOT = next((p for p in [_HERE, *_HERE.parents]
             if (p / 'adamacs' / 'ingest' / 'cellreg_plan.py').exists()),
            pathlib.Path(os.path.expanduser('~/adamacs_ingest')))
sys.path.insert(0, str(ROOT))

# ── Credentials and the schema prefix ───────────────────────────────────────
# dj_local_conf.json is NOT in the same place on every host: tatchu3 has it only at
# ~/adamacs/dj_local_conf.json, ibehavegpu1 also has one in the repo root. Load it BEFORE
# importing anything from adamacs -- the prefix is read at import, and without credentials
# DataJoint prompts for a username on stdin.
for _cand in (os.environ.get('DJ_LOCAL_CONF'),
              os.path.expanduser('~/adamacs/dj_local_conf.json'),
              str(ROOT / 'dj_local_conf.json'),
              'dj_local_conf.json'):
    if _cand and os.path.exists(_cand):
        dj.config.load(_cand); print('dj config:', _cand); break
else:
    raise FileNotFoundError('no dj_local_conf.json found; set DJ_LOCAL_CONF')

import adamacs
if not str(pathlib.Path(adamacs.__file__).resolve()).startswith(str(ROOT.resolve())):
    raise ImportError(
        f"'adamacs' resolved to {adamacs.__file__}\ninstead of {ROOT}. Another checkout is "
        f"installed in this environment. RESTART THE KERNEL and run this cell first -- "
        f"sys.path cannot displace an already-imported package.")
print('adamacs package:', os.path.dirname(adamacs.__file__))

# Import the schema THROUGH THE PIPELINE. `from adamacs.schemas import cell_registration`
# gives you the module but leaves it UNACTIVATED -- CellRegRun.database comes back None and
# every query raises "'NoneType' object has no attribute ...". pipeline.py is what calls
# activate(). Note this import performs a small INSERT (a Lab row, skip_duplicates), so it
# cannot run on a read-only connection.
from adamacs.pipeline import cell_registration as cr, imaging, session, scan
import pathlib, subprocess, collections
from IPython.display import Image, display

FIG = ROOT / "notebooks" / "figures"      # gitignored; regenerated below
FIG.mkdir(parents=True, exist_ok=True)
conn = dj.conn()
assert cr.CellRegRun.database, "schema not activated -- import via adamacs.pipeline"
print("schema:", cr.CellRegRun.database)

## 1. The workflow, start to finish

The diagram below is generated from the repository, not drawn by hand. Read it top to
bottom: a task says *what to register*, the exporter *builds the input files from the
database*, MATLAB *does the matching*, and then the result is *registered and parsed back
in*.

Colours: amber = files and existing production tables · green = tables this schema adds ·
blue = tables filled automatically by `populate()` · red = the one path that could ever
write production, which stays unused.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "scripts" / "draw_cell_registration_diagrams.py"),
                str(FIG)], check=True, cwd=str(ROOT))
display(Image(filename=str(FIG / "workflow_cell_registration.png"), width=760))

## 2. The same thing as a table graph

Now the tables themselves. A **box inside a rounded frame** is a *part table* — a table that
cannot exist without its master and is deleted with it. An arrow means "needs a row over
there first".

`imaging.Curation` is the amber box: it is the **only** existing production table this
schema touches, and only ever by pointing at it. Nothing upstream is modified.

In [ ]:
display(Image(filename=str(FIG / "erd_cell_registration.png"), width=980))

### And straight from the live server

The diagram above is parsed from the source. This one is `dj.Diagram` asking the database
what is actually declared — useful when you want to be sure the server matches the code.

Two quirks are worth knowing if you draw these yourself: `make_dot()` raises
*"Class DISKModel is not properly declared"* unless you clear the diagram's `context`, and
DataJoint writes node labels as HTML (`<<u>`schema`.`table`</u>>`) that needs stripping to
be readable.

In [ ]:
import re

def _clean(raw):
    txt = re.sub(r"<[^>]*>", "", str(raw).strip().strip('"')).strip("<>").replace("`", "")
    return txt.replace("roselab_cell_registration.", "").replace("roselab_", "").strip(" .")

def draw(diagram, name, size='"22,26"'):
    diagram.context = {}                      # else make_dot() trips over adamacs.schemas.disk
    dot = diagram.make_dot()
    dot.set("size", size); dot.set("dpi", "160")
    for n in dot.get_nodes():
        lbl = _clean(n.get_label() or "")
        if lbl:
            n.set_label(f'"{lbl}"'); n.set_fontname("Helvetica"); n.set_fontsize("11")
            if "__" in lbl:                   # a part table -> box it
                n.set_shape("box"); n.set_style("filled"); n.set_fillcolor("#f2f2f2")
                n.set_fontsize("9.5")
    out = FIG / name
    dot.write_png(str(out))
    return out

display(Image(filename=str(draw(dj.Diagram(cr.schema), "dj_cellreg_schema.png")), width=980))

## 3. The tables in plain words

**Settings — what was asked for**

| Table | One row per | In plain words |
|---|---|---|
| `MatchingMethod` | method | Which algorithm: probabilistic, simple threshold, or `cellreg_unknown` when the run has no log and the approach is not recoverable. |
| `CellRegParamSet` | distinct settings | The knobs: distance limit, `p_same` threshold, microns per pixel. Keyed by a hash of its own contents, so identical settings are one row. |

**Planning — what you intend to register**

| Table | One row per | In plain words |
|---|---|---|
| `RegistrationSite` | field of view | A patch of brain followed over days. Named by the site's *first* session id. |
| `CellRegTask` | planned run | "Register these sessions of this site, with these settings, into this folder." |
| `CellRegTask.Session` | **column** | **The manifest.** Column 0 is this curation, column 1 is that one… This is the table that fixes the old problem — see §4. |

**Building the input**

| Table | One row per | In plain words |
|---|---|---|
| `CellRegExport` | task | What was written to disk, when, by which code version. |
| `CellRegExport.File` | column | One `Fall_<NN>_<scan>.mat`, its checksum, its mask count, and the curation it came from. |

**The run that happened**

| Table | One row per | In plain words |
|---|---|---|
| `CellRegRun` | result file | Identified by the **sha256 of the file**, not its path — scan folders hold byte-identical backups. |
| `CellRegRun.Session` | column | Which curation each column turned out to be, and *how we know* (`resolution_method`). |
| `CellRegRun.UnresolvedColumn` | column we could not pin down | With the reason, in words. Honest gaps rather than guesses. |
| `CellRegRun.Quality` | run | The accuracy numbers from the log: false positives, uncertain pairs. |
| `CellRegRun.Copy` | duplicate file | Other paths holding the same bytes. |

**What CellReg actually found**

| Table | One row per | In plain words |
|---|---|---|
| `CellRegResult` | run | Summary: how many cells, how many pairs, whether `p_same` was readable. |
| `CellRegResult.Cell` | **cell** | One tracked neuron. No session in its key — that is what makes it cross-session. |
| `CellRegResult.Member` | cell × session | "Cell 7 is mask 412 in this scan." Joins `Segmentation.Mask` with no renaming. |
| `CellRegResult.PairQuality` | cell × session pair | `p_same`: how sure the match between those two sessions is. |
| `MaskCoordinates` | mask in a run | Where each mask sits, before and after alignment. |

**Afterwards**

| Table | One row per | In plain words |
|---|---|---|
| `RegistrationCuration` / `.CellDecision` | human pass / cell | Someone accepted or rejected a cell by hand. Kept, never overwritten. |
| `StableCell` / `StableCellMap` | identity across *runs* | Lets a cell keep an id when the site is re-registered later. |
| `RoiGroupProjection` / `.Scan` | projection | What `roi_group` **would** become. Writing it needs explicit authorisation; it has never been run. |
| `RoiGroupSnapshot` / `.Mask` | snapshot | What `roi_group` held *before*, so a projection is reversible. |

In [ ]:
rows = []
for name in sorted(dir(cr)):
    obj = getattr(cr, name)
    if not (hasattr(obj, "full_table_name") and not name.startswith("_")):
        continue
    rows.append((name, len(obj())))
    for sub in ("Session", "Cell", "Member", "PairQuality", "Copy", "Quality",
                "UnresolvedColumn", "CellDecision", "Scan", "Mask", "File"):
        part = getattr(obj, sub, None)
        if part is not None and hasattr(part, "full_table_name"):
            rows.append((f"{name}.{sub}", len(part())))
print(f"{'table':<38}{'rows':>12}")
for n, k in sorted(rows, key=lambda r: -r[1]):
    print(f"{n:<38}{k:>12,}")
print(f"\n{len(rows)} definitions")

## 4. How task insertion works

This is the heart of it, so here it is slowly.

### The problem it fixes

The old export copied `<ScanPath>/suite2p/plane0/Fall.mat`. That path contains a *scan* —
but a scan can have **many curations** (one of ROS-2101's scans has seven), and the path does
not say which one. So after the fact nobody could tell which curation a column was built
from. Worse, 696 of the 743 multi-curation groups in the lab have **byte-identical masks**,
so you cannot work it out by comparing shapes either. The information was never recorded,
and it is not recoverable.

### The fix: say it first, in a table

`CellRegTask.Session` has `column_index` **in its primary key** and a foreign key to
`imaging.Curation`. In words: *"column 0 of this task is curation 7 of scan X."* The order
is what the MATLAB driver consumes, and the curation is a declared value, not a guess.

A unique index stops the same curation being used twice in one task, and
`CellRegExport.make()` refuses a task whose `column_index` values are not 0, 1, 2, … without
gaps — a gap would silently shift every later column against `cell_to_index_map`.

### What you actually call

```python
keys = [ {'session_id': …, 'scan_id': …, 'paramset_idx': 10, 'curation_id': 7}, …  ]  # ORDERED
task = cr.CellRegTask.auto_generate_entries(site_key, keys, paramset=pset,
                                            task_mode='trigger', output_dir=str(out),
                                            label='Gratings')
```

`auto_generate_entries` writes one `CellRegTask` row plus one `CellRegTask.Session` row per
entry in `keys`, numbering them by their position in the list. Nothing is written to disk
here — it only plans. `task_mode='trigger'` means "build the inputs from the database";
`'load'` means "a run already exists in this folder, just record it".

### Then the manifest is carried all the way through

1. `CellRegExport.populate()` reads the declared curations' masks and writes one
   `Fall_<NN>_<scan>.mat` per column — so the file genuinely *is* that curation.
2. MATLAB runs and writes `Results/cellRegistered_*.mat`.
3. `register_results` finds the task whose `cellreg_output_dir` contains that file, reads
   the manifest back, and writes `CellRegRun.Session` with
   **`resolution_method = 'manifest'`** — the only value that means *declared* rather than
   *reconstructed afterwards*.

If a column's centroid count disagrees with its declared curation's mask count, that step
**raises** instead of quietly falling back to a weaker method: the exporter built that
column from those masks, so a mismatch means the task and the folder describe different
exports, and guessing would defeat the whole point.

In [ ]:
task_cols = (cr.CellRegTask.Session * cr.CellRegTask.proj('task_label', 'task_mode')).fetch(
    order_by="subject, cellreg_task_id, column_index", as_dict=True)
print("the manifest, as actually stored:\n")
print(f"{'subject':<10}{'site':<15}{'task':>5}{'col':>5}  {'scan':<15}{'ps':>4}{'cur':>5}  label")
for t in task_cols:
    print(f"{t['subject']:<10}{t['same_site_id']:<15}{t['cellreg_task_id']:>5}"
          f"{t['column_index']:>5}  {t['scan_id']:<15}{t['paramset_idx']:>4}"
          f"{t['curation_id']:>5}  {t['task_label']}")

print("\nand how every column in the database was resolved:")
for m, n in sorted(collections.Counter(
        cr.CellRegRun.Session.fetch("resolution_method")).items(), key=lambda kv: -kv[1]):
    print(f"   {n:>5}  {m}")

`manifest` is the only one of those that is provenance. The others were reconstructed
after the fact and are ranked by how much they prove:

- **`roi_group_match`** — evidence. The column's implied identities match one curation's
  stored `roi_group` exactly and uniquely.
- **`mask_set_identical`** — a convention. Several curations matched and were byte-identical
  in masks, so the choice cannot change any data — only which `curation_id` is recorded.
- **`log_file`** — weak. The scan had exactly one curation, so the column was forced. One
  candidate existing is not evidence it is the right one.

## 5. What this schema never does

It never writes `roi_group`. Cross-session identity lives in `CellRegResult.Cell` /
`.Member`; `roi_group` stays a *derived projection* in `RoiGroupProjection`, which records
the run and the `cell_index` behind each value. No stage of the worker calls it, and it has
0 rows.

In [ ]:
print("RoiGroupProjection:", len(cr.RoiGroupProjection()),
      "| .Scan:", len(cr.RoiGroupProjection.Scan()),
      "| RoiGroupSnapshot:", len(cr.RoiGroupSnapshot()))
assert len(cr.RoiGroupProjection()) == 0, "a projection has been run -- check with Tobias"
print("\nnothing in this schema has ever written roi_group.")

---

**Next:** notebook 17 parses runs that already exist and shows how to query a tracked cell's
responses; notebook 18 triggers a new run from one config cell. The planner that chooses the
columns is `adamacs/ingest/cellreg_plan.py`. The wiki page
`Database/Schemas/roselab_cell_registration.md` carries the measured state and the traps.